In [ ]:
import os

In [ ]:
os.chdir("../")

In [ ]:
%pwd

In [ ]:
import torch
import torch_geometric.data.data
import torch_geometric.data.storage

# Add this before you call torch.save()
torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.storage.GlobalStorage
])

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [ ]:
from src.frauddetection.constants import *
from src.frauddetection.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )

        return data_transformation_config

In [ ]:
import os
import torch
from src.frauddetection import logger
from torch_geometric.datasets import EllipticBitcoinDataset

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def initiate_data_transformation(self):
        try:
            # Load the dataset from the ingestion artifacts
            dataset = EllipticBitcoinDataset(root=self.config.data_path)
            data = dataset[0]

            # 1. Binary Labeling (Class 1 = Illicit, Class 2 -> 0 = Licit)
            # Original: 1=illicit, 2=licit, 0=unknown
            y_binary = data.y.clone()
            y_binary[data.y == 2] = 0 
            y_binary[data.y == 1] = 1
            
            # 2. Temporal Splitting Logic (80/20 split)
            # Time step is the first feature in the Elliptic dataset
            node_times = data.x[:, 0]
            train_mask = node_times < 35  # First 34 time steps for training
            test_mask = node_times >= 35   # Remaining for testing
            
            # 3. Filter for labeled nodes only (removing 'unknown' class 0)
            labeled_mask = data.y != 0
            data.train_mask = train_mask & labeled_mask
            data.test_mask = test_mask & labeled_mask
            data.y_binary = y_binary

            logger.info("Successfully applied temporal splitting and binary labeling.")
            
            # Save the processed object or masks
            torch.save(data, os.path.join(self.config.root_dir, "transformed_data.pt"))
            
        except Exception as e:
            raise e

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    raise e